In [1]:
import numpy as np
import torch

In [2]:
def scaled_dot_product_Attention(Q, K, V):
    Q = np.array(Q)
    K = np.array(K)
    V = np.array(V)
    d_k = K.shape[-1]

    scores = (Q @ np.transpose(K, axes=(0, 2, 1))) / np.sqrt(d_k)
    max_scores = np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores - max_scores)

    weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

    result = weights @ V

    return (result, weights)

### Эксперименты: подготовка


In [3]:
print(f"NumPy {np.__version__}; seed=42")
rng = np.random.default_rng(42)
Q = rng.normal(size=(2, 4, 8))
K = rng.normal(size=(2, 4, 8))
V = rng.normal(size=(2, 4, 6))


NumPy 2.5.3; seed=42


### 1. Влияние масштаба Q и K на веса


In [4]:
print("\n1. Масштаб Q и K: средний максимальный вес и энтропия строки (наты)")
for scale in (0.01, 1.0, 10.0):
    result, weights = scaled_dot_product_Attention(Q * scale, K * scale, V)
    assert np.isfinite(result).all() and np.isfinite(weights).all()
    np.testing.assert_allclose(weights.sum(axis=-1), 1)
    log_weights = np.zeros_like(weights)
    np.log(weights, out=log_weights, where=weights > 0)
    entropy = -(weights * log_weights).sum(axis=-1).mean()
    print(f"scale={scale:g}: mean(max weight)={weights.max(axis=-1).mean():.6f}, entropy={entropy:.6f}")



1. Масштаб Q и K: средний максимальный вес и энтропия строки (наты)
scale=0.01: mean(max weight)=0.250011, entropy=1.386294
scale=1: mean(max weight)=0.369125, entropy=1.306279
scale=10: mean(max weight)=0.998581, entropy=0.007890


### 2. Большие scores: устойчивый и обычный softmax


In [5]:
print("\n2. Большие scores: сравнение с нестабильным softmax")
# QKᵀ / √1 = [1000, 1001, 1002, 1003]; V=I позволяет видеть веса в result.
q = np.ones((1, 1, 1))
k = np.arange(1000, 1004, dtype=float).reshape(1, 4, 1)
v = np.eye(4)[None, :, :]
with np.errstate(over="raise", invalid="raise", divide="raise"):
    result, weights = scaled_dot_product_Attention(q, k, v)
    _, shifted_weights = scaled_dot_product_Attention(q, k - 1000, v)
    np.testing.assert_allclose(weights, shifted_weights)
    np.testing.assert_allclose(result, weights)
    np.testing.assert_allclose(weights.sum(axis=-1), 1)
print("Stable weights:", weights.ravel())
try:
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        exp_scores = np.exp(q @ k.transpose(0, 2, 1))
        naive_weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
except FloatingPointError as error:
    print("Без вычитания максимума:", error)
else:
    raise AssertionError(f"Ожидалось переполнение exp, получено {naive_weights}")



2. Большие scores: сравнение с нестабильным softmax
Stable weights: [0.0320586  0.08714432 0.23688282 0.64391426]
Без вычитания максимума: overflow encountered in exp


### 3. Underflow экспонент и граница устойчивости


In [6]:
print("\n3. Underflow exp и переполнение QKᵀ — разные случаи")
k = np.array([0.0, -1000.0, -2000.0, -3000.0]).reshape(1, 4, 1)
with np.errstate(over="raise", invalid="raise", divide="raise", under="ignore"):
    result, weights = scaled_dot_product_Attention(q, k, v)
np.testing.assert_array_equal(weights.ravel(), [1, 0, 0, 0])
np.testing.assert_array_equal(result, weights)
print("Underflow малых экспонент, конечные weights:", weights.ravel())
with np.errstate(over="ignore"):
    overflowing_scores = (q * 1e200) @ (np.ones((1, 4, 1)) * 1e200).transpose(0, 2, 1)
assert np.isposinf(overflowing_scores).all()
print("QKᵀ при входах 1e200:", overflowing_scores.ravel())
try:
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        scaled_dot_product_Attention(q * 1e200, np.ones((1, 4, 1)) * 1e200, v)
except FloatingPointError as error:
    print("Переполнение до softmax:", error)
else:
    raise AssertionError("Ожидалось переполнение матричного произведения")

print("\nВсе ожидаемые результаты экспериментов подтверждены.")



3. Underflow exp и переполнение QKᵀ — разные случаи
Underflow малых экспонент, конечные weights: [1. 0. 0. 0.]
QKᵀ при входах 1e200: [inf inf inf inf]
Переполнение до softmax: invalid value encountered in subtract

Все ожидаемые результаты экспериментов подтверждены.


## 4. Сравнение моей функции с PyTorch

Сравниваем готовые Q, K, V без обучаемых проекций, масок и dropout, на CPU в float64.
Моя функция возвращает **кортеж `(result, weights)`**, а
[`torch.nn.functional.scaled_dot_product_attention`](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.scaled_dot_product_attention.html)
возвращает **только тензор результата**.

Для PyTorch добавляем одну голову: `(B, L, D) → (B, 1, L, D)`, затем убираем её из результата.
Веса в Torch отдельно вычисляем через `torch.softmax` для проверки: это не дополнительный выход библиотечной функции.
В каждом примере печатаются все входы, оба фактических возврата, формы и максимальная абсолютная разница.


In [7]:
import torch.nn.functional as F

print(f"NumPy {np.__version__}, PyTorch {torch.__version__}; CPU, float64")


def compare_with_torch(Q, K, V):
    Q = np.asarray(Q, dtype=np.float64)
    K = np.asarray(K, dtype=np.float64)
    V = np.asarray(V, dtype=np.float64)
    print("Q =\n", Q)
    print("K =\n", K)
    print("V =\n", V)

    numpy_output = scaled_dot_product_Attention(Q, K, V)
    numpy_result, numpy_weights = numpy_output
    q_t, k_t, v_t = [torch.from_numpy(x).unsqueeze(1) for x in (Q, K, V)]
    with torch.no_grad():
        torch_output = F.scaled_dot_product_attention(
            q_t, k_t, v_t, attn_mask=None, dropout_p=0.0, is_causal=False
        )
        torch_weights = torch.softmax(
            (q_t @ k_t.transpose(-2, -1)) / np.sqrt(Q.shape[-1]), dim=-1
        ).squeeze(1)
    torch_result = torch_output.squeeze(1)

    print("\nМоя функция возвращает (result, weights):\n", numpy_output)
    print("\nF.scaled_dot_product_attention возвращает Tensor (с осью головы):\n", torch_output)
    print("\nРезультат Torch после squeeze(1):\n", torch_result)
    print("\nВеса, отдельно вычисленные torch.softmax:\n", torch_weights)
    print("\nФормы NumPy: result", numpy_result.shape, "weights", numpy_weights.shape)
    print("Формы Torch: output", tuple(torch_output.shape), "result", tuple(torch_result.shape))
    print("Суммы моих весов:\n", numpy_weights.sum(axis=-1))
    print("max |result NumPy − Torch| =", np.max(np.abs(numpy_result - torch_result.numpy())))
    print("max |weights NumPy − torch.softmax| =", np.max(np.abs(numpy_weights - torch_weights.numpy())))

    assert numpy_result.shape == (Q.shape[0], Q.shape[1], V.shape[-1])
    assert numpy_weights.shape == (Q.shape[0], Q.shape[1], K.shape[1])
    assert np.isfinite(numpy_result).all() and np.isfinite(numpy_weights).all()
    assert (numpy_weights >= 0).all()
    np.testing.assert_allclose(numpy_weights.sum(axis=-1), 1.0, rtol=1e-12, atol=1e-12)
    np.testing.assert_allclose(numpy_result, torch_result.numpy(), rtol=1e-12, atol=1e-12)
    np.testing.assert_allclose(numpy_weights, torch_weights.numpy(), rtol=1e-12, atol=1e-12)
    print("PASS: результаты и веса совпадают с допуском rtol=atol=1e-12.")

NumPy 2.5.3, PyTorch 2.14.1; CPU, float64


### 4.1. Маленький пример с заданными значениями
Два query, три key/value; d_k=2 и d_v=2. Числа небольшие, удобно проследить смешивание V.


In [8]:
Q_small = np.array([[[1., 0.], [0., 1.]]])
K_small = np.array([[[1., 0.], [0., 1.], [1., 1.]]])
V_small = np.array([[[10., 0.], [0., 20.], [30., 30.]]])
compare_with_torch(Q_small, K_small, V_small)

Q =
 [[[1. 0.]
  [0. 1.]]]
K =
 [[[1. 0.]
  [0. 1.]
  [1. 1.]]]
V =
 [[[10.  0.]
  [ 0. 20.]
  [30. 30.]]]

Моя функция возвращает (result, weights):
 (array([[[16.04448371, 15.98887907],
        [14.01112093, 20.05560463]]]), array([[[0.40111209, 0.19777581, 0.40111209],
        [0.19777581, 0.40111209, 0.40111209]]]))

F.scaled_dot_product_attention возвращает Tensor (с осью головы):
 tensor([[[[16.0445, 15.9889],
          [14.0111, 20.0556]]]], dtype=torch.float64)

Результат Torch после squeeze(1):
 tensor([[[16.0445, 15.9889],
         [14.0111, 20.0556]]], dtype=torch.float64)

Веса, отдельно вычисленные torch.softmax:
 tensor([[[0.4011, 0.1978, 0.4011],
         [0.1978, 0.4011, 0.4011]]], dtype=torch.float64)

Формы NumPy: result (1, 2, 2) weights (1, 2, 3)
Формы Torch: output (1, 1, 2, 2) result (1, 2, 2)
Суммы моих весов:
 [[1. 1.]]
max |result NumPy − Torch| = 1.7763568394002505e-15
max |weights NumPy − torch.softmax| = 0.0
PASS: результаты и веса совпадают с допуском rtol=

### 4.2. Нулевой Q: равномерные веса и среднее V
Используем те же K и V. Три key означают вес 1/3 для каждого; каждый query получает среднее V.


In [9]:
compare_with_torch(np.zeros_like(Q_small), K_small, V_small)

Q =
 [[[0. 0.]
  [0. 0.]]]
K =
 [[[1. 0.]
  [0. 1.]
  [1. 1.]]]
V =
 [[[10.  0.]
  [ 0. 20.]
  [30. 30.]]]

Моя функция возвращает (result, weights):
 (array([[[13.33333333, 16.66666667],
        [13.33333333, 16.66666667]]]), array([[[0.33333333, 0.33333333, 0.33333333],
        [0.33333333, 0.33333333, 0.33333333]]]))

F.scaled_dot_product_attention возвращает Tensor (с осью головы):
 tensor([[[[13.3333, 16.6667],
          [13.3333, 16.6667]]]], dtype=torch.float64)

Результат Torch после squeeze(1):
 tensor([[[13.3333, 16.6667],
         [13.3333, 16.6667]]], dtype=torch.float64)

Веса, отдельно вычисленные torch.softmax:
 tensor([[[0.3333, 0.3333, 0.3333],
         [0.3333, 0.3333, 0.3333]]], dtype=torch.float64)

Формы NumPy: result (1, 2, 2) weights (1, 2, 3)
Формы Torch: output (1, 1, 2, 2) result (1, 2, 2)
Суммы моих весов:
 [[1. 1.]]
max |result NumPy − Torch| = 0.0
max |weights NumPy − torch.softmax| = 0.0
PASS: результаты и веса совпадают с допуском rtol=atol=1e-12.


### 4.3. Случайные тензоры: два batch, разные длины и d_v ≠ d_k
Q=(2,2,3), K=(2,3,3), V=(2,3,2). Фиксированный seed=123; печатаем оба batch полностью.


In [10]:
comparison_rng = np.random.default_rng(123)
Q_random = comparison_rng.normal(size=(2, 2, 3))
K_random = comparison_rng.normal(size=(2, 3, 3))
V_random = comparison_rng.normal(size=(2, 3, 2))
compare_with_torch(Q_random, K_random, V_random)

Q =
 [[[-0.98912135 -0.36778665  1.28792526]
  [ 0.19397442  0.9202309   0.57710379]]

 [[-0.63646365  0.54195222 -0.31659545]
  [-0.32238912  0.09716732 -1.52593041]]]
K =
 [[[ 1.1921661  -0.67108968  1.00026942]
  [ 0.13632112  1.53203308 -0.65996941]
  [-0.31179486  0.33776913 -2.2074711 ]]

 [[ 0.82792144  1.54163039  1.12680679]
  [ 0.75476964 -0.14597789  1.28190223]
  [ 1.07403062  0.39262084  0.00511431]]]
V =
 [[[-0.36176687 -1.2302322 ]
  [ 1.22622929 -2.17204389]
  [-0.37014735  0.16438007]]

 [[ 0.85988118  1.76166124]
  [ 0.99332378 -0.29152143]
  [ 0.72812756 -1.26160032]]]

Моя функция возвращает (result, weights):
 (array([[[-0.01211044, -1.27600978],
        [ 0.46911245, -1.50373323]],

       [[ 0.84953031,  0.25402294],
        [ 0.81008581, -0.3582068 ]]]), array([[[0.66291211, 0.22080089, 0.116287  ],
        [0.31814734, 0.52405777, 0.1577949 ]],

       [[0.42164693, 0.24830384, 0.33004923],
        [0.23751242, 0.1910477 , 0.57143988]]]))

F.scaled_dot_product_

### 4.4. Большие scores
Scores=[1000,1001,1002,1003]. Сравниваем устойчивые реализации на тех же входах.


In [11]:
compare_with_torch(
    np.ones((1, 1, 1)),
    np.arange(1000, 1004, dtype=np.float64).reshape(1, 4, 1),
    np.array([[[1., 0.], [0., 1.], [2., 0.], [0., 2.]]]),
)

Q =
 [[[1.]]]
K =
 [[[1000.]
  [1001.]
  [1002.]
  [1003.]]]
V =
 [[[1. 0.]
  [0. 1.]
  [2. 0.]
  [0. 2.]]]

Моя функция возвращает (result, weights):
 (array([[[0.50582424, 1.37497284]]]), array([[[0.0320586 , 0.08714432, 0.23688282, 0.64391426]]]))

F.scaled_dot_product_attention возвращает Tensor (с осью головы):
 tensor([[[[0.5058, 1.3750]]]], dtype=torch.float64)

Результат Torch после squeeze(1):
 tensor([[[0.5058, 1.3750]]], dtype=torch.float64)

Веса, отдельно вычисленные torch.softmax:
 tensor([[[0.0321, 0.0871, 0.2369, 0.6439]]], dtype=torch.float64)

Формы NumPy: result (1, 1, 2) weights (1, 1, 4)
Формы Torch: output (1, 1, 1, 2) result (1, 1, 2)
Суммы моих весов:
 [[1.]]
max |result NumPy − Torch| = 2.220446049250313e-16
max |weights NumPy − torch.softmax| = 1.1102230246251565e-16
PASS: результаты и веса совпадают с допуском rtol=atol=1e-12.
